DEPLOYMENT NLP CON FASTAPI

Fast API standard per mettere in produzione Modelli di Deep Learning.
Immagina di dover gestire migliaia di messaggi al secondo, non puoi permeterti che il server si fermi a pensare ad ogni virgola. Oggi vediamo come trasformare il modello NLP in un arma veloce, sicura e autodocumentata

- Definizione di endpoint: utilizzo di Pydantic e Hints per la validazione automatica dei dati
- Il paradigma asincrono: gestire I/O non bloccante per massimizzare la reattività del server
- Ingegneria della risposta: strutturazione dei dati in uscita e documentazione automatica con Swagger UI

Come Fast API progette il modello dal caos dell'input umano

FastAPI e la Validazione dei Dati
Creare interfaccie robuste per l'upload di testo
Mentre Flask si basa su un approccio flessibile ma manuale (dovevamo controllare manualmente la qualità del dato inviata dall'utente), FastAPI sfrutta i moderni Type Hints di Python per automatizzare la validazione. Il server sa esattamente cosa aspettarsi e rifiuta qualsiasi dati che non rispetta le caratteristiche richieste. Questo significa che il server rifiuta automaticamente richieste malformate prima ancora che raggiungano il modella NLP.
Utilizzeremo 'Pydantic' per definire questi contratti (degli schemi di dati chiari), permettendo al sistema di mappare un corpo JSON direttamente in un oggetto Python tipizzato, riducendo drasticamente il codice boilerplate.

Quali sono gli ingrananggi tecnici che permettono tutto ciò?

Architettura dell'Endpoint
I componenti chiave sono:
- BaseModel di Pydantic: classe per definire la struttura del payload atteso, includendo vincoli di lunghezza o formato del testo. Pydantic è la nostra porta in ingresso, se il json non entra nella sagoma prevista, FastAPI solleva un muro.
- Type Hints: annotazioni che permettono a FastAPI di generare automaticamente la documentazione e i controlli di tipo
- Dependency Injection: sistema per gestire risorse condivise, come il caricamente del modello o le chiavi API, in modo modulare. Immagina di dover condividere lo stesso modello con 10 funzioni diverse, invece di caricarlo ogni volta fastAPI lo ignetta dove serve, sfrttando le risorse in modo intelligente.
- Path e Query Parameters: metodi per passare metadati aggiuntivi all'inferenza senza appesantire il corpo della richiesta.

Cosa succede quando un utente prova ad inviare un dato sbagliato?

Dall'Input al Modello
Se un client invia un numero dove un modello NLP aspetta una stringa (dato sbagliato), FastAPI restituisce un errore 422 'Unprocecssable Entity' in modo autonomo, proteggendo il modello da input invalidi (validazione all'ingresso). Non dovete scrivere una sola riga di codice per gestire questo eerrore.
Pydantic converte i dati in tempi rapidissimi grazie alla sua implementazione in Rust, garantendo che la fase di validazione non diventi un collo di bottiglia (Data Parsing efficiente).
In progetti di grandi dimensioni, utilizzeremo 'APIRouter' per separare gli endpoint di classificazione da quelli di traduzione o summarization.

Possiamo tradurre questa logica di filtro in un concetto matematico semplice

Modellazione della Validazione
Possiamo vedere la validazione come la funzione di appartenenza ad un dominio.
Se l'input x apppartiene allo schema che abbiamo definito, il sistema restituisce 1 ed i dati vanno nei neuroni del sistema, altrimenti restituisce 0 ed il sistema si blocca.
La validazione può essere vistas come un filtro che garantisce la conformità del dato in ingresso rispetto alla specifiche del modello neurale.
Possiamo esprimere il controllo di integrità come una funzione che ammette solo elementi appartenenti al dominio testuale definito dallo schema.

Questa integrita del dat è la prima linea di difesa contro i crash del sistema.

Ora che l'ingresso è sicuro dobbiamo occuparci di come il server gestisce il tempo

Programmazione Asincrona per NLP
Gestire il concurrency model di FastAPI
Immagina un ristorante con un solo chef che può cucinare un solo piatto alla volta. Se deve bollire le uova per 10 minuti, tutti i clienti aspettano affamati. Il paradigma asincrono di FastAPI cambia tutto. le chef mette l'acqua sul fiuco e mentra aspetta prepare gli antipasti.
In NLP caricare i dati o chiamare un database, richiede tempo di attesa, grazie alla asinc i/o il nostro server può gestire più cliente contemporaneamente restando sempre reattivo.
FastAPI è costruito su "starlette" e supporta nativamente il paradigma 'asyncio'. Questo permette al server di gestire migliaia di connessioni simultanee mentre attendo operazioni I/O, come il caricamento di dati da un database.
Tuttavia, l'inferenza NLP è un'operazione CPU-intensive. Vedremo come bilanciare l'uso di 'async def' per la gestione web e l'esecuzione sincrona per i calcoli pesanti del modello.

Ma quali sono i comandi per pilotare questo chef multitasking?

Async ed Event Loop
Meccanismi di non-blocing I/O
- Async def: definisce uan funzione coroutine che può sospendere l'esecuzione durante l'attesa di dati esterni.
- Await: operatore che indica al server di procedere con altre richieste mentre l'operazione corrente è in pausa. Stiamo dicendo al server, mentri aspetti che questa operazione finisca, vai pure ad aiutare altri.
- Starlette Background Task: funzionalità per eseguire task post-risposta, come il logging o il salvataggio dei risultati nel DB
- Uvicorn: server ASGI che gestisce il ciclo degli eventi ad alte prestazioni per fastAPI

Ma attenzione, non tutto può essere asincrono, c'è una trappola da evitare

Performance e Blococ del Server
Dobbiamo distinguere tra compiti I/O bound (aspettare la rete) e compiti CPU bound (calcolare i tensori). l'inferenza di un modello Deep Learning è un lavoro enorme per la CPU, se la facciamo dentro un loop asincrono senza attenzione, lo chef si fermerà di nuovo.
FastAPI da questo punto di vista è intelligente con endpoin come def o come async def
Le chiamate database o API esterne sono I/O bound e traggono massimo beneficio da 'async'. L'inferenza tensoriale è CPU bound e richiede attenzione specifica (I/O bound vs CPU bound)
Se definiamo un endpoint come 'def' invece di 'async def', FastAPI lo esegue automaticamente in un thread separato per non bloccare l'event loop principale (esecuzione in Thread Pool), è come avere una corsia preferenziale per le auto veloci ed una zona di parcheggio per i lavori pesanti.
Per scalare realmente l'inferenza, useremo più worker Uvicorn, ciascuno con il proprio processo, per saturare tutti i core della macchina.

Vediamo come questo impatta sulla capacità totale del sistema

Efficienza del Sistema Asincrono
Modellazione del throughput 
L'uso dell'asincronia permette di aumentare il numero di richieste gestire contemporaneamente senza aumentare proporzionalmente le risorse hardware.
Il throughtput del server può essere analizzato in funzione del tempo medio di attesa e dalla capacità di gestione simultanea dell'event loop

Ora che il motore corre veloce dobbiamo assicurarci che la carozzeria sia impeccabile

Gestione della Risposta e Documentazione
Restituire insight NLP in modo professionale
Una volta completata l'infernza, il risultato deve essere serializzato in una risposta JSON strutturata. FastAPI garantisce che l'output segua lo schema promesso, fornendo coerenza agli sviluppatori front-end
Uno dei vantaggi competitivi di FastAPI è la generazione automatica di una documentazione interattiva. Ogni endpoint creato è immediatamente testabile tramite una UI web, rendendo il debugging e l'integrazione estremamente rapidi

Standard di Output
Dall'inferenza alla consegna del dato
- Response Model: utilizzo di Pydantic per filtrare e formattare i dati in uscita, nascondendo log sensibili o dettagli tecnici
- Status Codes: implementazione corretta di codice HTTP come 200 (successo), 201 (creato) o 503 (modello non disponibile)
- OpenAPI Spec: standard aperto utilizzato da FastAPI per descrivere le aPI, permettendo la generazione automatica di client in altri linguaggi
- Interactive Docs: accesso a '/docs' per testare l'inferenza del modello direttamente dal browser senza strumenti esterni.

Usabiltà e Manutebilità
Possiamo intercettare errori specifii del modello (es. testo troppo lungo) e restituire messaggi JSOM personalizzati invece di un generico errore di server (Custom Exception Handling)
La configurazione del Cross-Origin Resource Sharing è essenziale per permettere alla nostra dashboard web di interrogare l'API da un dominio diverso (CORS e Sicurezza).
FastAPI gestisce automaticamente la conversione di oggetti complessi come timestamp o array NumPy nel formato JSON standard.

Risposta finale

Mapping Semantico dei Risultati
Strutturazione dei logit
La risposta dell'API deve tradurre i valori numerici grezzi del modello in etichette semantiche comprensibii per l'utente finale.
La probabilità viene solitamente normalizzata tramite softmax e inviata come parte del dizionario di rispostas alla classe predetta.

In [1]:
# python -m pip install fastapi
import os

# Configurazione dell'ambiente per disabilitare l'uso della GPU e impostare Torch come motore di calcolo per Keras.
# Questo garantisce la stabilità del servizio su server o ambienti senza hardware grafico dedicato.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["KERAS_BACKEND"] = "torch"

import keras
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from contextlib import asynccontextmanager

# Definizione e registrazione della classe SentimentWrapper.
# Questa classe deve essere definita nello stesso script (o importata) dove avviene il caricamento
# affinché Keras possa deserializzare correttamente il modello salvato in formato .keras.
@keras.saving.register_keras_serializable(package="MyCustomModels")
class SentimentWrapper(keras.Model):
    """
    Incapsula un modello Hugging Face per renderlo compatibile con le API di Keras.
    """
    def __init__(self, model_id="distilbert-base-uncased-finetuned-sst-2-english", **kwargs):
        super().__init__(**kwargs)
        self.model_id = model_id
        # Caricamento locale del modello per l'inferenza.
        from transformers import AutoModelForSequenceClassification
        self.hf_model = AutoModelForSequenceClassification.from_pretrained(model_id)

    def call(self, inputs):
        """
        Esegue il passaggio in avanti (forward pass) dei dati attraverso il modello.
        """
        return self.hf_model(**inputs).logits

    def get_config(self):
        """
        Restituisce i parametri necessari per ricostruire l'istanza del wrapper.
        """
        return {"model_id": self.model_id}

# Definizione dello schema della richiesta API tramite Pydantic.
# Questo assicura la validazione automatica del formato della richiesta in ingresso.
class SentimentRequest(BaseModel):
    text: str

# Dizionario per mantenere i modelli in memoria durante il ciclo di vita dell'applicazione.
ml_models = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    """
    Gestisce l'inizializzazione e la chiusura dell'applicazione.
    Le risorse pesanti (come i modelli pesanti) vengono caricate all'avvio e mantenute in memoria.
    """
    base_path = os.path.dirname(os.path.abspath(__file__))
    model_path = os.path.join(base_path, "sentiment_model.keras")
    
    print(f"Caricamento delle risorse dal percorso: {base_path}")
    try:
        # Caricamento del modello Keras precedentemente salvato.
        ml_models["model"] = keras.models.load_model(model_path)
        # Caricamento del tokenizer salvato localmente.
        ml_models["tokenizer"] = AutoTokenizer.from_pretrained(base_path)
        print("Risorse caricate correttamente.")
    except Exception as e:
        print(f"Errore critico durante il caricamento delle risorse: {e}")
    
    yield
    # Pulizia delle risorse allo spegnimento del server.
    ml_models.clear()

# Inizializzazione dell'applicazione FastAPI con gestione del lifespan.
app = FastAPI(title="Analisi del Sentiment API", lifespan=lifespan)

@app.post("/predict")
def predict(request: SentimentRequest):
    """
    Endpoint per l'analisi del sentiment di un testo.
    
    Flusso di elaborazione:
    1. Ricezione del testo e trasformazione in vettori numerici (Tokenizzazione).
    2. Conversione dei dati nel formato compatibile con il modello.
    3. Inferenza tramite il modello per ottenere i logit.
    4. Trasformazione dei logit in probabilità e selezione della classe dominante.
    """
    model = ml_models.get("model")
    tokenizer = ml_models.get("tokenizer")
    
    if model is None or tokenizer is None:
        raise HTTPException(status_code=503, detail="Modello non inizializzato.")
    
    # 1. Fase di Tokenizzazione: trasforma il testo in tensori leggibili dal modello (PyTorch).
    tokens = tokenizer(request.text, return_tensors="pt")
    
    # 2. Preparazione Input: convertiamo l'output del tokenizer in un dizionario standard
    # per garantire la compatibilità con l'interfaccia di chiamata di Keras.
    model_inputs = dict(tokens) 
    
    with torch.no_grad():
        # 3. Inferenza: il modello processa i dati e restituisce i valori grezzi di output (logits).
        logits = model(model_inputs) 
        
        # 4. Post-processing: applichiamo la funzione Softmax per ottenere una distribuzione di probabilità.
        # Questo trasforma i logit in valori compresi tra 0 e 1 che sommano a 1.
        probabilities = torch.nn.functional.softmax(logits, dim=-1).numpy()[0]
    
    # Individuazione dell'indice con la probabilità più alta (0 = Negativo, 1 = Positivo).
    prediction_index = np.argmax(probabilities)
    
    return {
        "text": request.text,
        "label": "POSITIVE" if prediction_index == 1 else "NEGATIVE",
        "confidence": float(probabilities[prediction_index])
    }

if __name__ == "__main__":
    import uvicorn
    # Avvio del server web locale sulla porta 8000.
    uvicorn.run(app, port=8000)

c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: asyncio.run() cannot be called from a running event loop